# Asistente Virtual DUOC UC Plaza Oeste
Implementación de Arquitectura RAG con LangChain, FAISS y GitHub Models.

## Objetivo
Diseñar un asistente virtual basado en IA capaz de responder consultas académicas utilizando documentación institucional oficial.

In [ ]:
# INSTALAMOS EL PIP
!pip install langchain langchain-openai langchain-community faiss-cpu tiktoken pypdf python-dotenv

In [2]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    base_url=os.getenv("GITHUB_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN")
)

In [3]:
# IMPORTACIONES IMPORTANTES

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
import os

c:\Users\ignac\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# NOS CONECTAMS CON EL MODELO Y HACEMOS UNA PRUEBA

from openai import OpenAI
import os

client = OpenAI(
    base_url=os.environ["GITHUB_BASE_URL"],
    api_key=os.environ["GITHUB_TOKEN"]
)

response = client.chat.completions.create(
    model="gpt-4o-mini",  # modelo disponible en GitHub Models
    messages=[
        {"role": "system", "content": "Eres un asistente académico."},
        {"role": "user", "content": "Explica qué es Rag en una frase."}
    ],
    temperature=0.2
)

print(response.choices[0].message.content)

Rag es un término que puede referirse a una tela o trapo usado para limpiar, pero también puede hacer referencia a un tipo de música o a un estilo de arte en contextos específicos.


In [ ]:
# INTALAMOS LOS EMBEDDINGS

!pip install sentence-transformers

In [5]:
# IMPORTAMOS DATA

import os

print(os.listdir("data"))

['admisiones_2026.pdf', 'becas.pdf', 'calendario_2026.pdf', 'faq.pdf', 'reglamento_academico.pdf']


In [6]:
# CARGAMOS LOS DOCUMENTOS CREADOS EN DATA

from langchain_community.document_loaders import PyPDFLoader
import os

def load_documents(data_path="data"):
    documents = []
    
    for file in os.listdir(data_path):
        if file.endswith(".pdf"):
            print(f"Cargando {file}...")
            loader = PyPDFLoader(os.path.join(data_path, file))
            documents.extend(loader.load())
    
    return documents

docs = load_documents()
print(f"\nSe cargaron {len(docs)} páginas en total.")

Cargando admisiones_2026.pdf...
Cargando becas.pdf...
Cargando calendario_2026.pdf...
Cargando faq.pdf...
Cargando reglamento_academico.pdf...

Se cargaron 92 páginas en total.


In [10]:
# DIVIDIMOS EN CHUNKS

from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=400,
        chunk_overlap=80
    )
    return text_splitter.split_documents(documents)

chunks = split_documents(docs)

print(f"Se generaron {len(chunks)} fragmentos.")

Se generaron 617 fragmentos.


In [11]:
# NOS ASEGURAMOS QUE LA API KEY ESTE FUNCIONADO BIEN
import os
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")

In [14]:
# CREAMOS EMBEDDINGS Y BASE VECTORIAL (FAISS)

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

def create_vectorstore_batched(chunks, batch_size=20):
    embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=os.getenv("GITHUB_TOKEN"),
    openai_api_base=os.getenv("GITHUB_BASE_URL")
)
    
    vectorstore = None
    
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i+batch_size]
        print(f"Procesando batch {i//batch_size + 1}...")
        
        if vectorstore is None:
            vectorstore = FAISS.from_documents(batch, embeddings)
        else:
            vectorstore.add_documents(batch)
    
    return vectorstore

vectorstore = create_vectorstore_batched(chunks, batch_size=20)

print("Base vectorial creada correctamente.")

print("Base vectorial creada correctamente.")

Procesando batch 1...
Procesando batch 2...
Procesando batch 3...
Procesando batch 4...
Procesando batch 5...
Procesando batch 6...
Procesando batch 7...
Procesando batch 8...
Procesando batch 9...
Procesando batch 10...
Procesando batch 11...
Procesando batch 12...
Procesando batch 13...
Procesando batch 14...
Procesando batch 15...
Procesando batch 16...
Procesando batch 17...
Procesando batch 18...
Procesando batch 19...
Procesando batch 20...
Procesando batch 21...
Procesando batch 22...
Procesando batch 23...
Procesando batch 24...
Procesando batch 25...
Procesando batch 26...
Procesando batch 27...
Procesando batch 28...
Procesando batch 29...
Procesando batch 30...
Procesando batch 31...
Base vectorial creada correctamente.
Base vectorial creada correctamente.


In [15]:
# PROBAMOS SI ENTREGA INFORMACIÓN DE UN DOCUMENTO

query = "¿Cómo puedo realizar mi matrícula?"

results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print(f"\nResultado {i+1}:\n")
    print(doc.page_content[:500])


Resultado 1:

1. ¿Cómo puedo realizar mi proceso de matrícula? 
El proceso de matrícula se realiza a través del portal institucional en línea. El estudiante 
debe ingresar con su RUT y contraseña, completar los datos solicitados, aceptar el 
contrato de prestación de servicios educacionales y efectuar el pago correspondiente 
según las modalidades disponibles.

Resultado 2:

según las modalidades disponibles. 
 
2. ¿Cuáles son los medios de pago disponibles para la matrícula y arancel? 
Los pagos pueden realizarse mediante: 
 Tarjeta de débito.  
 Tarjeta de crédito.  
 Transferencia bancaria.  
 Pago presencial en caja institucional (según disponibilidad). 
 
3. ¿Dónde puedo consultar el calendario académico oficial?

Resultado 3:

4 
 
TÍTULO IV 
DE LA MATRÍCULA 
 
Artículo N°10. Se denomina Matrícula, a la inscripción oficial del/de la estudiante en los registros 
académicos de Duoc UC, adquiriendo la calidad de Alumno/a Regular o Provisional por primera vez 
o por renovación, 

In [16]:
# CREAMOS UN PROMPT OFICIAL

from langchain_core.prompts import PromptTemplate

template = """
Eres un asistente virtual del Instituto DUOC UC Plaza Oeste.
Debes responder exclusivamente utilizando la información entregada en el contexto.
Si la información no está presente, indica que dichos datos estan fuera de tu área de atención.
No inventes información.
Usa lenguaje formal y claro.

Contexto:
{context}

Pregunta del usuario:
{question}

Respuesta:
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

In [18]:
# CONECTAMOS EL LLM

from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3,
    openai_api_key=os.getenv("GITHUB_TOKEN"),
    openai_api_base=os.getenv("GITHUB_BASE_URL")
)

In [19]:
# VEMOS LA FUNCIÓN DEL RAG

def rag_answer(question):
    
    # 1. Recuperar documentos relevantes
    docs = vectorstore.similarity_search(question, k=3)
    
    # 2. Construir contexto
    context = "\n\n".join([doc.page_content for doc in docs])
    
    # 3. Formatear prompt
    formatted_prompt = prompt.format(
        context=context,
        question=question
    )
    
    # 4. Llamar al modelo
    response = llm.invoke(formatted_prompt)
    
    return response.content

In [20]:
# HACEMOS UNA PRUEBA DEL SISTEMA COMPLETO

respuesta = rag_answer("¿Cómo puedo realizar mi matrícula?")
print(respuesta)

El proceso de matrícula se realiza a través del portal institucional en línea. Debe ingresar con su RUT y contraseña, completar los datos solicitados, aceptar el contrato de prestación de servicios educacionales y efectuar el pago correspondiente según las modalidades disponibles.


In [21]:
# HACEMOS UNA PREGUNTA FUERA DE CONTEXTO PARA VALIDAR ERRORES

respuesta = rag_answer("¿Cuál es la capital de Francia?")
print(respuesta)

Dicha información está fuera de mi área de atención.


In [22]:
# GUARDAMOS LA BASE VECTORIAL

vectorstore.save_local("faiss_index")

In [25]:
# PARA CARGAR LA BASE VECTORIAL A FUTURO

import os
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=os.getenv("GITHUB_TOKEN"),
    openai_api_base=os.getenv("GITHUB_BASE_URL")
)

vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)
